In [0]:
from pyspark.sql.functions import (
    col, from_json, explode, current_date, date_sub, 
    when, sqrt, pow, lit, coalesce, count, sum as _sum, avg, countDistinct
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, ArrayType, BooleanType
)
from delta.tables import DeltaTable

# Read all raw player stats data (for initial load)
# For incremental processing, filter by: col("createdate") >= date_sub(current_date(), 7)
raw_df = spark.read.table("workspace.fotmob.raw_player_stats")

print(f"Processing {raw_df.count()} raw records from the last 7 days...")

# Check for records with shotmap data
raw_df_with_shots = raw_df.filter(
    col("raw_json").contains('"shotmap"')
)

print(f"Records with shotmap data: {raw_df_with_shots.count()}")
display(raw_df_with_shots.select("player_id", "season", "competition", "createdate").limit(5))

Processing 8822 raw records from the last 7 days...
Records with shotmap data: 8410


player_id,season,competition,createdate
99767,2019,NWSL,2026-06-07
99767,2021,NWSL,2026-06-07
99767,2023,NWSL,2026-06-07
99767,2025,NWSL,2026-06-07
99818,2019,NWSL,2026-06-07


In [0]:
# Define schema for shotmap array
# Note: Use LongType for IDs since they exceed 32-bit integer range
from pyspark.sql.types import LongType

onGoalShot_schema = StructType([
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("zoomRatio", DoubleType(), True)
])

shot_schema = StructType([
    StructField("id", LongType(), True),  # Changed to LongType for large IDs
    StructField("eventType", StringType(), True),
    StructField("teamId", IntegerType(), True),
    StructField("playerId", IntegerType(), True),
    StructField("playerName", StringType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("min", IntegerType(), True),
    StructField("minAdded", IntegerType(), True),
    StructField("isBlocked", BooleanType(), True),
    StructField("isOnTarget", BooleanType(), True),
    StructField("blockedX", DoubleType(), True),
    StructField("blockedY", DoubleType(), True),
    StructField("goalCrossedY", DoubleType(), True),
    StructField("goalCrossedZ", DoubleType(), True),
    StructField("expectedGoals", DoubleType(), True),
    StructField("expectedGoalsOnTarget", DoubleType(), True),
    StructField("shotType", StringType(), True),
    StructField("situation", StringType(), True),
    StructField("period", StringType(), True),
    StructField("isOwnGoal", BooleanType(), True),
    StructField("onGoalShot", onGoalShot_schema, True),
    StructField("box", StringType(), True),
    StructField("matchId", IntegerType(), True),
    StructField("matchDate", StringType(), True)
])

shotmap_schema = StructType([
    StructField("shotmap", ArrayType(shot_schema), True)
])

# Parse JSON and explode shotmap array to get one row per shot
df_shots = raw_df_with_shots.select(
    col("player_id"),
    col("season"),
    col("competition"),
    col("createdate"),
    from_json(col("raw_json"), shotmap_schema).alias("parsed")
).select(
    col("player_id"),
    col("season"),
    col("competition"),
    col("createdate"),
    explode(col("parsed.shotmap")).alias("shot")
).select(
    col("player_id"),
    col("season"),
    col("competition"),
    col("createdate"),
    col("shot.id").alias("shot_id"),
    col("shot.matchId").alias("match_id"),
    col("shot.matchDate").alias("match_date"),
    col("shot.teamId").alias("team_id"),
    col("shot.playerName").alias("player_name"),
    col("shot.x").alias("shot_x"),
    col("shot.y").alias("shot_y"),
    col("shot.eventType").alias("event_type"),
    col("shot.shotType").alias("shot_type"),
    col("shot.situation").alias("situation"),
    col("shot.box").alias("box_location"),
    col("shot.min").alias("minute"),
    col("shot.minAdded").alias("minute_added"),
    col("shot.period").alias("period"),
    col("shot.isOnTarget").alias("is_on_target"),
    col("shot.isBlocked").alias("is_blocked"),
    col("shot.isOwnGoal").alias("is_own_goal"),
    col("shot.expectedGoals").alias("expected_goals"),
    col("shot.expectedGoalsOnTarget").alias("expected_goals_on_target"),
    col("shot.onGoalShot.x").alias("on_goal_shot_x"),
    col("shot.onGoalShot.y").alias("on_goal_shot_y"),
    col("shot.blockedX").alias("blocked_x"),
    col("shot.blockedY").alias("blocked_y"),
    col("shot.goalCrossedY").alias("goal_crossed_y"),
    col("shot.goalCrossedZ").alias("goal_crossed_z")
)

# Calculate shot distance from goal (assuming goal is at x=100, y=50 based on typical shot map coordinates)
df_shots = df_shots.withColumn(
    "shot_distance",
    sqrt(pow(lit(100) - col("shot_x"), 2) + pow(lit(50) - col("shot_y"), 2))
)

print(f"Total individual shots extracted: {df_shots.count()}")
print(f"Unique players with shots: {df_shots.select('player_id').distinct().count()}")
display(df_shots.limit(10))

Total individual shots extracted: 15727
Unique players with shots: 938


player_id,season,competition,createdate,shot_id,match_id,match_date,team_id,player_name,shot_x,shot_y,event_type,shot_type,situation,box_location,minute,minute_added,period,is_on_target,is_blocked,is_own_goal,expected_goals,expected_goals_on_target,on_goal_shot_x,on_goal_shot_y,blocked_x,blocked_y,goal_crossed_y,goal_crossed_z,shot_distance
99767,2025,NWSL,2026-06-07,2791246865,4719658,2025-03-23T19:00:00Z,521233,Sophie Schmidt,98.2456140347,37.756,AttemptSaved,LeftFoot,FromCorner,InsideBox,20,null,FirstHalf,true,true,false,0.0976547524333,0.0,0.536044973544974,0.322751321164021,101.5862068968,37.05,35.75375,1.219999994,12.369050331987562
99767,2025,NWSL,2026-06-07,2823764273,4719733,2025-06-21T23:30:00Z,521233,Sophie Schmidt,88.1779411764,27.110238095,Miss,RightFoot,FromCorner,OutsideBox,75,null,SecondHalf,false,false,false,0.054205022752285,0.0,1.54507618215052,0.677248677248677,null,null,29.631666665,5.427586216,25.762419818337094
99767,2025,NWSL,2026-06-07,2831523497,4719740,2025-08-03T02:00:00Z,521233,Sophie Schmidt,101.681034483,33.31375,Goal,Header,FromCorner,InsideBox,88,null,SecondHalf,true,false,false,0.62960159778595,0.981070280075073,1.32275132275132,0.504511275714286,null,null,32.78,1.9070526222,16.770713043741914
99767,2025,NWSL,2026-06-07,2832800575,4719743,2025-08-09T00:47:00Z,521233,Sophie Schmidt,101.9655172416,34.305,Goal,LeftFoot,RegularPlay,InsideBox,90,5,SecondHalf,true,false,false,0.555245637893677,0.982050776481628,0.798280423280423,0.0968253963492064,null,null,34.7625,0.3659999982,15.817594097302752
99767,2025,NWSL,2026-06-07,2844051619,4719777,2025-09-08T00:30:00Z,521233,Sophie Schmidt,91.7,24.5597452279,Miss,RightFoot,SetPiece,InsideBox,90,5,SecondHalf,false,false,false,0.0343080200254917,0.0,2.0,0.349914153142513,null,null,24.349554145,3.3768275888,26.75998062161774
99767,2025,NWSL,2026-06-07,2844051799,4719777,2025-09-08T00:30:00Z,521233,Sophie Schmidt,88.5,25.7071875,Miss,RightFoot,RegularPlay,OutsideBox,90,7,SecondHalf,false,false,false,0.129611641168594,0.0,0.0,0.0476192499440481,null,null,45.1918471296,0.5329473658,26.87732760451002
99818,2025,NWSL,2026-06-07,2788050211,4719646,2025-03-15T00:00:00Z,728922,Marta,89.9,40.721666667,Miss,LeftFoot,RegularPlay,InsideBox,17,null,FirstHalf,false,false,false,0.0698212161660194,0.0,2.0,0.0473513452339569,null,null,20.84636943,0.6228421022,13.714863084925739
99818,2025,NWSL,2026-06-07,2788066073,4719646,2025-03-15T00:00:00Z,728922,Marta,80.5703883504,45.3319745182,AttemptSaved,LeftFoot,RegularPlay,OutsideBox,57,null,SecondHalf,true,true,false,0.0385550111532211,0.0,0.73776455026455,0.322751321164021,96.9912280694,38.368333335,34.99125,1.219999994,19.982499111797964
99818,2025,NWSL,2026-06-07,2788070475,4719646,2025-03-15T00:00:00Z,728922,Marta,84.3995145632,37.5075,Miss,LeftFoot,FreeKick,OutsideBox,76,null,SecondHalf,false,false,false,0.137621209025383,0.0,0.636374381660523,0.677248677248677,null,null,36.135,3.9764137976,19.98593760907426
99818,2025,NWSL,2026-06-07,2791316017,4719657,2025-03-23T21:00:00Z,728922,Marta,94.1929824545,40.553571429,AttemptSaved,LeftFoot,RegularPlay,InsideBox,26,null,FirstHalf,true,false,false,0.0609026066958904,0.248366430401802,1.26223544973545,0.0543581172486773,100.732758621,37.672,33.00875,0.2054736832,11.088573646810934


In [0]:
from pyspark.sql.functions import when, split
from pyspark.sql.types import IntegerType

# Parse season: 
# - For tournaments (EURO, World Cup) with slashed format: take first year
# - For other slashed formats (leagues): take second year
# - Otherwise: convert to int
df_shots = df_shots.withColumn(
    "season",
    when(
        col("season").contains("/") & 
        (col("competition").contains("EURO") | col("competition").contains("World Cup")),
        split(col("season"), "/")[0].cast(IntegerType())
    ).when(
        col("season").contains("/"),
        split(col("season"), "/")[1].cast(IntegerType())
    ).otherwise(
        col("season").cast(IntegerType())
    )
)

print(f"Season values after formatting:")
df_shots.select("season", "competition").distinct().orderBy("season", "competition").show(50, truncate=False)

Season values after formatting:
+------+------------------------+
|season|competition             |
+------+------------------------+
|2022  |UEFA Women's EURO       |
|2025  |NWSL                    |
|2025  |UEFA Women's EURO       |
|2025  |UEFA Women's EURO Grp. A|
|2025  |UEFA Women's EURO Grp. B|
|2025  |UEFA Women's EURO Grp. C|
|2025  |UEFA Women's EURO Grp. D|
|2025  |Women's FA Cup          |
|2026  |A-League Women          |
|2026  |NWSL                    |
|2026  |Premiere Ligue          |
|2026  |Supercup der Frauen     |
|2026  |WSL                     |
|2026  |Women's FA Cup          |
+------+------------------------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Deduplicate: keep only the most recent record for each shot_id
window_spec = Window.partitionBy("shot_id").orderBy(col("createdate").desc())

df_shots_deduped = df_shots.withColumn(
    "row_num", row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num", "createdate")

print(f"After deduplication: {df_shots_deduped.count()} unique shots")
print(f"Shot type breakdown:")
df_shots_deduped.groupBy("shot_type").count().orderBy(col("count").desc()).show()
print(f"Event type breakdown:")
df_shots_deduped.groupBy("event_type").count().orderBy(col("count").desc()).show()

After deduplication: 15681 unique shots
Shot type breakdown:
+--------------+-----+
|     shot_type|count|
+--------------+-----+
|     RightFoot| 8885|
|      LeftFoot| 4358|
|        Header| 2416|
|OtherBodyParts|   22|
+--------------+-----+

Event type breakdown:
+------------+-----+
|  event_type|count|
+------------+-----+
|AttemptSaved| 7915|
|        Miss| 5647|
|        Goal| 1762|
|        Post|  357|
+------------+-----+



In [0]:
# Join shot data with goal classifications
df_shots_with_situation = df_shots_deduped.join(
    goals_classified.select(
        "shot_id",
        "is_go_ahead_goal",
        "is_leveller",
        "is_insurance_goal",
        "is_consolation_goal"
    ),
    "shot_id",
    "left"
)

# Aggregate by player_id, season, competition
df_player_stats = df_shots_with_situation.groupBy(
    "player_id", "season", "competition"
).agg(
    # Total shots
    count("*").alias("total_shots"),
    _sum(when(col("is_on_target") == True, 1).otherwise(0)).alias("shots_on_target"),
    _sum(when(col("is_on_target") == False, 1).otherwise(0)).alias("shots_off_target"),
    
    # Goals
    _sum(when(col("event_type") == "Goal", 1).otherwise(0)).alias("goals"),
    
    # Expected goals
    _sum(coalesce(col("expected_goals"), lit(0))).alias("total_xg"),
    _sum(coalesce(col("expected_goals_on_target"), lit(0))).alias("total_xgot"),
    
    # Shot type breakdown
    _sum(when(col("shot_type") == "LeftFoot", 1).otherwise(0)).alias("left_foot_shots"),
    _sum(when(col("shot_type") == "RightFoot", 1).otherwise(0)).alias("right_foot_shots"),
    _sum(when(col("shot_type") == "Header", 1).otherwise(0)).alias("header_shots"),
    _sum(when(col("shot_type").isNull() | ~col("shot_type").isin(["LeftFoot", "RightFoot", "Header"]), 1).otherwise(0)).alias("other_body_part_shots"),
    
    # Location breakdown
    _sum(when(col("box_location") == "InsideBox", 1).otherwise(0)).alias("inside_box_shots"),
    _sum(when(col("box_location") == "OutsideBox", 1).otherwise(0)).alias("outside_box_shots"),
    
    # Situation breakdown
    _sum(when(col("situation") == "RegularPlay", 1).otherwise(0)).alias("regular_play_shots"),
    _sum(when(col("situation") == "FromCorner", 1).otherwise(0)).alias("corner_shots"),
    _sum(when(col("situation") == "SetPiece", 1).otherwise(0)).alias("set_piece_shots"),
    _sum(when(col("situation").isNull() | ~col("situation").isin(["RegularPlay", "FromCorner", "SetPiece"]), 1).otherwise(0)).alias("other_situation_shots"),
    
    # Outcome breakdown
    _sum(when(col("event_type") == "Goal", 1).otherwise(0)).alias("goals_scored"),
    _sum(when(col("event_type") == "AttemptSaved", 1).otherwise(0)).alias("shots_saved"),
    _sum(when(col("event_type") == "Miss", 1).otherwise(0)).alias("shots_missed"),
    _sum(when(col("is_blocked") == True, 1).otherwise(0)).alias("shots_blocked"),
    
    # Average shot distance
    avg(col("shot_distance")).alias("avg_shot_distance"),
    
    # Goal situation breakdown
    _sum(when(col("is_go_ahead_goal") == True, 1).otherwise(0)).alias("go_ahead_goals"),
    _sum(when(col("is_leveller") == True, 1).otherwise(0)).alias("leveller_goals"),
    _sum(when(col("is_insurance_goal") == True, 1).otherwise(0)).alias("insurance_goals"),
    _sum(when(col("is_consolation_goal") == True, 1).otherwise(0)).alias("consolation_goals")
)

# Calculate derived metrics
df_player_stats = df_player_stats.withColumn(
    "shot_accuracy",
    when(col("total_shots") > 0, 
         col("shots_on_target") / col("total_shots") * 100
    ).otherwise(None)
).withColumn(
    "conversion_rate",
    when(col("total_shots") > 0,
         col("goals") / col("total_shots") * 100
    ).otherwise(None)
).withColumn(
    "xg_per_shot",
    when(col("total_shots") > 0,
         col("total_xg") / col("total_shots")
    ).otherwise(None)
)

print(f"Aggregated stats for {df_player_stats.count()} player/season/competition combinations")
display(df_player_stats.orderBy(col("goals").desc()).limit(10))

Aggregated stats for 1487 player/season/competition combinations


player_id,season,competition,total_shots,shots_on_target,shots_off_target,goals,total_xg,total_xgot,left_foot_shots,right_foot_shots,header_shots,other_body_part_shots,inside_box_shots,outside_box_shots,regular_play_shots,corner_shots,set_piece_shots,other_situation_shots,goals_scored,shots_saved,shots_missed,shots_blocked,avg_shot_distance,go_ahead_goals,leveller_goals,insurance_goals,consolation_goals,shot_accuracy,conversion_rate,xg_per_shot
971405,2026,WSL,118,70,48,21,20.648615710906686,20.00600489228964,21,54,43,0,111,7,85,19,2,12,21,49,45,19,17.010924064116615,8,1,11,1,59.32203389830508,17.796610169491526,0.17498826873649734
1214632,2026,Premiere Ligue,59,43,16,18,10.805222111761568,12.922213677316904,9,49,1,0,43,16,42,1,0,16,18,25,12,11,19.047299183165165,7,1,10,0,72.88135593220339,30.508474576271187,0.18313935782646726
1618199,2025,NWSL,59,45,14,15,14.167214278131723,13.012104172259571,3,53,3,0,52,7,41,2,2,14,15,30,13,12,17.85678419179323,9,0,6,0,76.27118644067797,25.423728813559322,0.2401222759005377
1400283,2026,Premiere Ligue,32,24,8,13,4.266570022329689,7.548872031271458,23,4,5,0,30,2,24,1,0,7,13,11,7,4,16.32170803855642,4,0,9,0,75.0,40.625,0.13333031319780278
1185212,2026,WSL,85,64,21,13,11.621242942444978,13.016958504915236,14,53,17,1,67,18,67,11,4,3,13,51,20,28,18.205055347207374,5,1,7,0,75.29411764705883,15.294117647058824,0.13672050520523504
857417,2025,NWSL,66,47,19,13,9.98449113344401,13.210269499570131,15,36,15,0,50,16,46,8,5,7,13,34,18,17,17.65415867491517,6,2,5,0,71.21212121212122,19.696969696969695,0.1512801686885456
1082557,2026,WSL,56,36,20,12,6.74831429310143,8.840030675753951,41,14,1,0,39,17,44,4,0,8,12,24,18,8,16.994812269071225,5,4,1,2,64.28571428571429,21.428571428571427,0.12050561237681125
1044266,2026,NWSL,61,38,23,12,7.2745217196643335,11.080185302533208,6,45,10,0,44,17,37,3,7,14,12,26,23,10,19.489835276836523,4,3,1,4,62.295081967213115,19.672131147540984,0.11925445442072678
1532180,2025,NWSL,63,46,17,11,11.594665841355917,12.216141613200305,17,38,8,0,45,18,54,3,0,6,11,35,16,13,18.84829614696217,5,2,3,1,73.01587301587301,17.46031746031746,0.18404231494215742
888653,2026,Premiere Ligue,63,39,24,11,9.94663038905561,9.760669589042664,13,49,1,0,44,19,48,3,2,10,11,28,24,10,20.869788895868158,3,1,5,2,61.904761904761905,17.46031746031746,0.15788302204850174


In [0]:
# Save shot_details table with shot_id primary key and liquid clustering
table_name = "workspace.fotmob.shot_details"

row_count = df_shots_deduped.count()
if row_count == 0:
    print(f"No new shot records to process in the last 7 days. Skipping write to preserve existing data.")
else:
    # If table doesn't exist, create it with liquid clustering
    if not spark.catalog.tableExists(table_name):
        df_shots_deduped.write.format("delta") \
            .mode("overwrite") \
            .option("delta.enableChangeDataFeed", "true") \
            .clusterBy("player_id", "season", "competition") \
            .saveAsTable(table_name)
        print(f"Created table {table_name} with {row_count} rows")
        print(f"Liquid clustering enabled on: player_id, season, competition")
    else:
        # Table exists - use MERGE to upsert records by shot_id
        delta_table = DeltaTable.forName(spark, table_name)
        delta_table.alias("target").merge(
            df_shots_deduped.alias("source"),
            "target.shot_id = source.shot_id"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print(f"Merged {row_count} shot records into {table_name} (updated existing or inserted new)")
    
    # Show final row count
    final_count = spark.read.table(table_name).count()
    print(f"Total rows in {table_name}: {final_count}")

Created table workspace.fotmob.shot_details with 15355 rows
Liquid clustering enabled on: player_id, season, competition
Total rows in workspace.fotmob.shot_details: 15355


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lag, lead, coalesce, lit

# Get all goals ordered by match and time
goals_with_timing = spark.read.table("workspace.fotmob.shot_details") \
    .filter(col("event_type") == "Goal") \
    .withColumn(
        "effective_minute",
        col("minute") + coalesce(col("minute_added"), lit(0))
    ) \
    .select(
        "shot_id",
        "match_id",
        "team_id",
        "player_id",
        "season",
        "competition",
        "effective_minute",
        "period"
    )

# Order goals within each match
window_match = Window.partitionBy("match_id").orderBy("effective_minute", "shot_id")

goals_ordered = goals_with_timing.withColumn(
    "goal_number", row_number().over(window_match)
)

# For each goal, calculate the score before it was scored
# We need to track both teams' scores
goals_with_prev_team = goals_ordered.withColumn(
    "prev_team_id", lag(col("team_id")).over(window_match)
)

# Calculate running score for each team
from pyspark.sql.functions import sum as _sum, when

window_running = Window.partitionBy("match_id").orderBy("effective_minute", "shot_id").rowsBetween(Window.unboundedPreceding, -1)

# For each goal, count goals scored by this team and by opponent before this goal
goals_with_score = goals_ordered.alias("current").join(
    goals_ordered.alias("previous"),
    (col("current.match_id") == col("previous.match_id")) &
    ((col("previous.effective_minute") < col("current.effective_minute")) |
     ((col("previous.effective_minute") == col("current.effective_minute")) &
      (col("previous.shot_id") < col("current.shot_id")))),
    "left"
).groupBy(
    col("current.shot_id"),
    col("current.match_id"),
    col("current.team_id"),
    col("current.player_id"),
    col("current.season"),
    col("current.competition"),
    col("current.effective_minute")
).agg(
    _sum(when(col("previous.team_id") == col("current.team_id"), 1).otherwise(0)).alias("team_goals_before"),
    _sum(when(col("previous.team_id") != col("current.team_id"), 1).otherwise(0)).alias("opp_goals_before")
)

# Replace null with 0 for first goal of match
goals_with_score = goals_with_score.withColumn(
    "team_goals_before", coalesce(col("team_goals_before"), lit(0))
).withColumn(
    "opp_goals_before", coalesce(col("opp_goals_before"), lit(0))
)

# Calculate scores after the goal
goals_with_score = goals_with_score.withColumn(
    "team_goals_after", col("team_goals_before") + 1
).withColumn(
    "opp_goals_after", col("opp_goals_before")
)

# Calculate goal difference before and after
goals_with_score = goals_with_score.withColumn(
    "goal_diff_before", col("team_goals_before") - col("opp_goals_before")
).withColumn(
    "goal_diff_after", col("team_goals_after") - col("opp_goals_after")
)

# Classify goal situation
goals_classified = goals_with_score.withColumn(
    "is_go_ahead_goal",
    # Was tied or losing, now winning
    (col("goal_diff_before") <= 0) & (col("goal_diff_after") > 0)
).withColumn(
    "is_leveller",
    # Was losing, now tied
    (col("goal_diff_before") < 0) & (col("goal_diff_after") == 0)
).withColumn(
    "is_insurance_goal",
    # Was already winning, extends lead to 2+
    (col("goal_diff_before") > 0) & (col("goal_diff_after") >= 2)
).withColumn(
    "is_consolation_goal",
    # Was losing, still losing after goal
    (col("goal_diff_before") < 0) & (col("goal_diff_after") < 0)
)

print(f"Classified {goals_classified.count()} goals by situation")

# Show sample classifications
print("\nSample goal classifications:")
goals_classified.select(
    "match_id",
    "player_id",
    "team_goals_before",
    "opp_goals_before",
    "team_goals_after",
    "opp_goals_after",
    "is_go_ahead_goal",
    "is_leveller",
    "is_insurance_goal",
    "is_consolation_goal"
).show(20, truncate=False)

# Verify classification counts
print("\nGoal situation breakdown:")
print(f"Go-ahead goals: {goals_classified.filter(col('is_go_ahead_goal')).count()}")
print(f"Levellers: {goals_classified.filter(col('is_leveller')).count()}")
print(f"Insurance goals: {goals_classified.filter(col('is_insurance_goal')).count()}")
print(f"Consolation goals: {goals_classified.filter(col('is_consolation_goal')).count()}")

Classified 1731 goals by situation

Sample goal classifications:
+--------+---------+-----------------+----------------+----------------+---------------+----------------+-----------+-----------------+-------------------+
|match_id|player_id|team_goals_before|opp_goals_before|team_goals_after|opp_goals_after|is_go_ahead_goal|is_leveller|is_insurance_goal|is_consolation_goal|
+--------+---------+-----------------+----------------+----------------+---------------+----------------+-----------+-----------------+-------------------+
|4719646 |1044266  |3                |0               |4               |0              |false           |false      |true             |false              |
|4892989 |1091618  |1                |0               |2               |0              |false           |false      |true             |false              |
|4892997 |1082541  |2                |1               |3               |1              |false           |false      |true             |false              |

In [0]:
# Save player_shot_stats table
table_name = "workspace.fotmob.player_shot_stats"

row_count = df_player_stats.count()
if row_count == 0:
    print(f"No new player shot stats to process in the last 7 days. Skipping write to preserve existing data.")
else:
    # If table doesn't exist, create it
    if not spark.catalog.tableExists(table_name):
        df_player_stats.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"Created table {table_name} with {row_count} rows")
    else:
        # Table exists - use MERGE to upsert records by player_id, season, competition
        delta_table = DeltaTable.forName(spark, table_name)
        delta_table.alias("target").merge(
            df_player_stats.alias("source"),
            "target.player_id = source.player_id AND target.season = source.season AND target.competition = source.competition"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print(f"Merged {row_count} player stat records into {table_name} (updated existing or inserted new)")
    
    # Show final row count
    final_count = spark.read.table(table_name).count()
    print(f"Total rows in {table_name}: {final_count}")
    
    # Display sample of top scorers
    print("\nTop 10 players by goals:")
    display(spark.read.table(table_name).orderBy(col("goals").desc()).limit(10))

Created table workspace.fotmob.player_shot_stats with 1487 rows
Total rows in workspace.fotmob.player_shot_stats: 1487

Top 10 players by goals:


player_id,season,competition,total_shots,shots_on_target,shots_off_target,goals,total_xg,total_xgot,left_foot_shots,right_foot_shots,header_shots,other_body_part_shots,inside_box_shots,outside_box_shots,regular_play_shots,corner_shots,set_piece_shots,other_situation_shots,goals_scored,shots_saved,shots_missed,shots_blocked,avg_shot_distance,go_ahead_goals,leveller_goals,insurance_goals,consolation_goals,shot_accuracy,conversion_rate,xg_per_shot
971405,2026,WSL,118,70,48,21,20.648615710906686,20.00600489228964,21,54,43,0,111,7,85,19,2,12,21,49,45,19,17.010924064116615,8,1,11,1,59.32203389830508,17.796610169491526,0.17498826873649734
1214632,2026,Premiere Ligue,59,43,16,18,10.805222111761568,12.922213677316904,9,49,1,0,43,16,42,1,0,16,18,25,12,11,19.047299183165165,7,1,10,0,72.88135593220339,30.508474576271187,0.18313935782646726
1618199,2025,NWSL,59,45,14,15,14.167214278131723,13.012104172259571,3,53,3,0,52,7,41,2,2,14,15,30,13,12,17.85678419179323,9,0,6,0,76.27118644067797,25.423728813559322,0.2401222759005377
857417,2025,NWSL,66,47,19,13,9.98449113344401,13.210269499570131,15,36,15,0,50,16,46,8,5,7,13,34,18,17,17.65415867491517,6,2,5,0,71.21212121212122,19.696969696969695,0.1512801686885456
1400283,2026,Premiere Ligue,32,24,8,13,4.266570022329689,7.548872031271458,23,4,5,0,30,2,24,1,0,7,13,11,7,4,16.32170803855642,4,0,9,0,75.0,40.625,0.13333031319780278
1185212,2026,WSL,85,64,21,13,11.621242942444978,13.016958504915236,14,53,17,1,67,18,67,11,4,3,13,51,20,28,18.205055347207374,5,1,7,0,75.29411764705883,15.294117647058824,0.13672050520523504
1082557,2026,WSL,56,36,20,12,6.74831429310143,8.840030675753951,41,14,1,0,39,17,44,4,0,8,12,24,18,8,16.994812269071225,5,4,1,2,64.28571428571429,21.428571428571427,0.12050561237681125
1044266,2026,NWSL,61,38,23,12,7.2745217196643335,11.080185302533208,6,45,10,0,44,17,37,3,7,14,12,26,23,10,19.489835276836523,4,3,1,4,62.295081967213115,19.672131147540984,0.11925445442072678
1807245,2026,Premiere Ligue,40,26,14,11,6.165265529885888,8.489524237811565,9,29,2,0,32,8,32,4,0,4,11,15,14,8,17.24522487103436,4,5,1,1,65.0,27.500000000000004,0.1541316382471472
1680285,2026,Premiere Ligue,37,20,17,11,4.629302393645049,6.573376405052841,7,29,1,0,21,16,27,1,0,9,11,9,15,2,22.420725279725477,6,0,5,0,54.054054054054056,29.72972972972973,0.12511628090932564
